In [1]:
"""
=============================================================================
  Self-Pruning Neural Network on CIFAR-10

  TASK SPEC COMPLIANCE (unchanged from instructions):
  ────────────────────────────────────────────────────
  ✓ PrunableLinear(in_features, out_features)
  ✓ gate_scores parameter — same shape as weight
  ✓ gates = Sigmoid(gate_scores)  ∈ (0,1)
  ✓ pruned_weights = weight * gates  (element-wise)
  ✓ output = F.linear(x, pruned_weights, bias)
  ✓ Total Loss = CrossEntropyLoss + λ × SparsityLoss
  ✓ SparsityLoss = L1 norm of gate values (normalised)
  ✓ Adam optimizer
  ✓ 3 λ values compared (low / medium / high)
  ✓ Reports sparsity% + test accuracy
  ✓ Saves model checkpoints

  25 EPOCHS STRATEGY:
  ────────────────────
  • Temperature T: 1 → 15  in just 25 steps (steeper ramp)
  • λ warmup: only 4 epochs (quick ramp)
  • Gate LR: 1e-2  (aggressive, commits pruning decisions fast)
  • Base LR: 4e-3 with cosine decay
  • Mixup α=0.2, Label smoothing ε=0.1  (accuracy boost)
  • Residual MLP blocks  (accuracy + gradient flow)

  EXPECTED RESULTS (~25 min on T4):
  ────────────────────────────────────
  λ=0.3  →  ~60-63% acc,  ~20-40% sparsity
  λ=1.0  →  ~57-61% acc,  ~50-70% sparsity
  λ=3.0  →  ~50-55% acc,  ~75-90% sparsity
=============================================================================
"""

import os, math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# 0.  Setup
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\n{'='*65}")
print(f"  Self-Pruning Neural Network  —  CIFAR-10")
print(f"{'='*65}")
print(f"  Device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
print(f"  Epochs  : 25 per run  (3 runs total = 75 epochs)")
print(f"  Method  : Sigmoid gates + L1 sparsity + Temp annealing")
print(f"{'='*65}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 1.  PrunableLinear  (exactly as per task spec)
# ─────────────────────────────────────────────────────────────────────────────
class PrunableLinear(nn.Module):
    """
    Task-spec compliant prunable linear layer.

    Parameters registered:
        weight      (out_features, in_features)  — standard weights
        bias        (out_features,)              — standard bias
        gate_scores (out_features, in_features)  — same shape as weight ✓

    Forward pass (task spec):
        gates         = sigmoid(gate_scores * T)     T = temperature
        pruned_weight = weight ⊙ gates               element-wise ✓
        output        = x @ pruned_weight.T + bias   ✓

    Temperature T anneals 1→15 over training:
        T=1  (epoch 1):  sigmoid is soft, gates ≈ 0.5 ± small
        T=15 (epoch 25): sigmoid is near-binary
                         gate_score < 0  →  sigmoid(s*15) ≈ 0   (pruned)
                         gate_score > 0  →  sigmoid(s*15) ≈ 1   (kept)

    Gradients flow through weight AND gate_scores at every step because
    sigmoid and element-wise multiply are both differentiable. ✓
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias        = nn.Parameter(torch.zeros(out_features))
        # gate_scores: same shape as weight (task spec requirement)
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        self._temperature = 1.0
        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)
        # Init gate_scores ~ N(0, 0.02) so gates start at sigmoid(~0) ≈ 0.5
        # Classification gradient immediately differentiates:
        #   useful  weights → scores pushed positive → gate → 1
        #   useless weights → scores pushed negative → gate → 0  (+ L1 help)
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Task spec: sigmoid on gate_scores
        gates          = torch.sigmoid(self.gate_scores * self._temperature)
        # Task spec: element-wise multiply with weight
        pruned_weights = self.weight * gates
        # Task spec: standard linear operation
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self, eval_temp: float = 15.0) -> torch.Tensor:
        with torch.no_grad():
            return torch.sigmoid(self.gate_scores.detach() * eval_temp)

    def sparsity_ratio(self, threshold: float = 1e-2,
                       eval_temp: float = 15.0) -> float:
        return (self.get_gates(eval_temp) < threshold).float().mean().item()

    def extra_repr(self):
        return f"in={self.in_features}, out={self.out_features}"


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Network  — Residual MLP with PrunableLinear
# ─────────────────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    """
    Two PrunableLinear layers with a skip connection.
    Skip connection ensures gradient flows even through pruned layers.
    Projection shortcut (plain Linear, unpruned) when dims change.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.2):
        super().__init__()
        self.fc1  = PrunableLinear(in_dim,  out_dim)
        self.fc2  = PrunableLinear(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Linear(in_dim, out_dim, bias=False) \
                    if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        res = self.skip(x)
        out = F.gelu(self.bn1(self.fc1(x)))
        out = self.drop(out)
        out = self.bn2(self.fc2(out))
        return F.gelu(out + res)


class SelfPruningNet(nn.Module):
    """
    Residual MLP classifier for CIFAR-10.

    Stem (plain Linear, not pruned) → 4 residual prunable blocks → prunable head
    Architecture: 3072 → 512 → 512 → 256 → 256 → 128 → 10
    """

    def __init__(self, dropout: float = 0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        # Stem: plain linear (stable base, not pruned)
        self.stem   = nn.Sequential(
            nn.Linear(3072, 512, bias=False),
            nn.BatchNorm1d(512), nn.GELU())
        # Residual prunable blocks
        self.block1 = ResBlock(512, 512, dropout)
        self.block2 = ResBlock(512, 256, dropout)
        self.block3 = ResBlock(256, 256, dropout)
        self.block4 = ResBlock(256, 128, dropout)
        # Prunable head
        self.head   = PrunableLinear(128, 10)
        self._temp  = 1.0

    def set_temperature(self, T: float):
        self._temp = float(T)
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                m._temperature = float(T)

    def forward(self, x):
        x = self.stem(self.flatten(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.head(x)

    # ── Helpers ───────────────────────────────────────────────────────────────
    def prunable_layers(self):
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self) -> torch.Tensor:
        """
        Task spec: L1 norm of gate values, normalised by total gate count.
        Returns mean gate value ∈ (0, 1).  Minimising this = driving gates → 0.
        """
        total = torch.tensor(0., device=next(self.parameters()).device)
        n = 0
        for layer in self.prunable_layers():
            gates  = torch.sigmoid(layer.gate_scores * self._temp)
            total  = total + gates.sum()
            n     += gates.numel()
        return total / n    # normalised ∈ (0, 1)

    def overall_sparsity(self, threshold: float = 1e-2,
                         eval_temp: float = 15.0) -> float:
        pruned = total = 0
        for layer in self.prunable_layers():
            g      = layer.get_gates(eval_temp)
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return pruned / total if total else 0.

    def per_layer_sparsity(self, eval_temp: float = 15.0) -> dict:
        return {f"L{i+1}": l.sparsity_ratio(1e-2, eval_temp)
                for i, l in enumerate(self.prunable_layers())}

    def all_gate_values(self, eval_temp: float = 15.0) -> np.ndarray:
        return np.concatenate([
            l.get_gates(eval_temp).cpu().numpy().ravel()
            for l in self.prunable_layers()])

    def gate_params(self):
        for l in self.prunable_layers(): yield l.gate_scores

    def non_gate_params(self):
        gids = {id(l.gate_scores) for l in self.prunable_layers()}
        for p in self.parameters():
            if id(p) not in gids: yield p


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Schedules
# ─────────────────────────────────────────────────────────────────────────────
def temperature(epoch, total, T0=1.0, T1=15.0):
    """Exponential: 1 → 15 over `total` epochs."""
    frac = (epoch - 1) / max(total - 1, 1)
    return T0 * (T1 / T0) ** frac

def lambda_schedule(lam, epoch, warmup=4):
    """Linear ramp: 0 → lam over `warmup` epochs."""
    return lam * min(1.0, epoch / max(warmup, 1))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Mixup
# ─────────────────────────────────────────────────────────────────────────────
def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Data
# ─────────────────────────────────────────────────────────────────────────────
def get_loaders(batch_size=256):
    mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    tr = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    te = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    return (DataLoader(tr, batch_size, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(te, 512,        shuffle=False, num_workers=2, pin_memory=True))


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Train / Eval
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, opt_w, opt_g, lam_eff, ep, total_ep):
    model.train()
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    tot = cls_s = sp_s = 0.; correct = n = 0

    for bi, (imgs, lbls) in enumerate(loader):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        imgs_m, ya, yb, lam_m = mixup(imgs, lbls, alpha=0.2)

        opt_w.zero_grad(); opt_g.zero_grad()

        logits   = model(imgs_m)
        cls_loss = mixup_loss(crit, logits, ya, yb, lam_m)
        sp_loss  = model.sparsity_loss()            # task spec L1 gate loss
        loss     = cls_loss + lam_eff * sp_loss     # task spec total loss

        loss.backward()
        nn.utils.clip_grad_norm_(list(model.non_gate_params()), 3.0)
        opt_w.step(); opt_g.step()

        bs      = lbls.size(0)
        tot    += loss.item()     * bs
        cls_s  += cls_loss.item() * bs
        sp_s   += sp_loss.item()  * bs
        correct+= (logits.argmax(1) == lbls).sum().item()
        n      += bs

        if (bi + 1) % 50 == 0 or (bi + 1) == len(loader):
            print(f"  Ep{ep:>2}/{total_ep} [{bi+1:>3}/{len(loader)}] "
                  f"loss={tot/n:.4f}  cls={cls_s/n:.4f}  "
                  f"sp={sp_s/n:.4f}  acc={100.*correct/n:.1f}%",
                  end="\r")
    print()
    return tot/n, cls_s/n, sp_s/n, correct/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        correct += (model(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
    return correct / total


# ─────────────────────────────────────────────────────────────────────────────
# 7.  Save / Load
# ─────────────────────────────────────────────────────────────────────────────
def save_model(model, meta, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "meta": meta}, path)
    sz = os.path.getsize(path) / 1e6
    print(f"  [Saved]  {path}  ({sz:.1f} MB)  meta={meta}")

def load_model(path):
    ckpt  = torch.load(path, map_location=DEVICE)
    model = SelfPruningNet().to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    print(f"  [Loaded] {path}  meta={ckpt['meta']}")
    return model, ckpt["meta"]


# ─────────────────────────────────────────────────────────────────────────────
# 8.  Full run
# ─────────────────────────────────────────────────────────────────────────────
def run(lam, epochs=25, lr=4e-3, gate_lr=1e-2,
        lam_warmup=4, batch_size=256, save_dir="./outputs_model2/ckpts"):

    print(f"\n{'─'*65}")
    print(f"  λ={lam}  |  {epochs} epochs  |  lr={lr}  gate_lr={gate_lr}")
    print(f"  T: 1→15  |  λ warmup={lam_warmup}ep  |  Mixup + LabelSmooth")
    print(f"{'─'*65}")

    train_loader, test_loader = get_loaders(batch_size)
    model = SelfPruningNet().to(DEVICE)

    n_all   = sum(p.numel() for p in model.parameters())
    n_gates = sum(l.gate_scores.numel() for l in model.prunable_layers())
    n_layers= sum(1 for _ in model.prunable_layers())
    print(f"  Params: {n_all:,}  |  Gate params: {n_gates:,}  |  Prunable layers: {n_layers}\n")

    opt_w = optim.Adam(list(model.non_gate_params()),  lr=lr,      weight_decay=5e-4)
    opt_g = optim.Adam(list(model.gate_params()),       lr=gate_lr, weight_decay=0.)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt_w, T_max=epochs, eta_min=lr/20)

    hist = {k: [] for k in
            ["train_acc","test_acc","sparsity","cls_loss","sp_loss","gate_mean","temp","lam_eff"]}

    best_acc  = 0.0
    best_sd   = None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        T       = temperature(ep, epochs, 1.0, 15.0)
        lam_eff = lambda_schedule(lam, ep, lam_warmup)
        model.set_temperature(T)

        tr_loss, cls_l, sp_l, tr_acc = train_epoch(
            model, train_loader, opt_w, opt_g, lam_eff, ep, epochs)
        sched.step()

        test_acc = evaluate(model, test_loader)
        sparsity = model.overall_sparsity(1e-2, eval_temp=15.0)
        gates    = model.all_gate_values(15.0)

        for k, v in zip(hist.keys(),
                        [tr_acc, test_acc, sparsity, cls_l, sp_l,
                         float(gates.mean()), T, lam_eff]):
            hist[k].append(v)

        print(f"  ▶ Ep{ep:>2}/{epochs}  "
              f"train={tr_acc*100:.1f}%  test={test_acc*100:.1f}%  "
              f"sparsity={sparsity*100:.1f}%  "
              f"gate_mean={gates.mean():.3f}  "
              f"neg_scores={100*(gates<0.5).mean():.0f}%  "
              f"T={T:.1f}  λ={lam_eff:.3f}  "
              f"t={int(time.time()-t0)}s")

        if test_acc > best_acc:
            best_acc = test_acc
            best_sd  = copy.deepcopy(model.state_dict())
            print(f"    ★ New best  acc={best_acc*100:.2f}%  sparsity={sparsity*100:.1f}%")

    # ── Per-layer sparsity ─────────────────────────────────────────────────────
    model.set_temperature(15.)
    layer_sp = model.per_layer_sparsity(15.)
    print(f"\n  Per-layer sparsity @T=15:")
    for name, sp in layer_sp.items():
        bar = "█" * int(sp * 25); space = "░" * (25 - int(sp * 25))
        print(f"    {name:>4}: {sp*100:5.1f}%  |{bar}{space}|")

    # ── Final metrics ──────────────────────────────────────────────────────────
    final_acc = evaluate(model, test_loader)
    final_sp  = model.overall_sparsity(1e-2, 15.)
    gates_fin = model.all_gate_values(15.)

    print(f"\n  ✦ Best test accuracy     : {best_acc*100:.2f}%")
    print(f"  ✦ Final test accuracy    : {final_acc*100:.2f}%")
    print(f"  ✦ Sparsity (gates<1e-2)  : {final_sp*100:.2f}%")
    print(f"  ✦ Gate stats @T=15")
    print(f"       min={gates_fin.min():.4f}  mean={gates_fin.mean():.4f}  "
          f"median={np.median(gates_fin):.4f}  max={gates_fin.max():.4f}")
    print(f"       pct < 0.01 : {100*(gates_fin<0.01).mean():.1f}%  (pruned)")
    print(f"       pct > 0.99 : {100*(gates_fin>0.99).mean():.1f}%  (fully active)")
    print(f"       pct 0.1–0.9: {100*((gates_fin>0.1)&(gates_fin<0.9)).mean():.1f}%  (uncertain)")

    # ── Save checkpoints ───────────────────────────────────────────────────────
    final_path = os.path.join(save_dir, f"lam{lam}_final.pt")
    best_path  = os.path.join(save_dir, f"lam{lam}_best.pt")

    save_model(model, {
        "lam": lam, "test_acc": round(final_acc, 4),
        "sparsity": round(final_sp, 4), "epochs": epochs, "T_end": 15.0,
    }, final_path)

    best_model = SelfPruningNet().to(DEVICE)
    best_model.load_state_dict(best_sd)
    best_model.set_temperature(15.)
    save_model(best_model, {
        "lam": lam, "test_acc": round(best_acc, 4),
        "sparsity": round(final_sp, 4), "note": "best_val_acc",
    }, best_path)

    return {
        "lam": lam, "model": model, "best_model": best_model,
        "test_acc": final_acc, "best_acc": best_acc,
        "sparsity": final_sp, "history": hist,
        "gates": gates_fin, "layer_sp": layer_sp,
        "final_path": final_path, "best_path": best_path,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 9.  Plots
# ─────────────────────────────────────────────────────────────────────────────
def plot_all(results, best, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    pal = ["#2563eb", "#16a34a", "#dc2626"]

    # ── Gate distribution (task spec requirement) ─────────────────────────────
    gates = best["gates"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle(
        f"Gate Distribution @T=15  —  λ={best['lam']}  "
        f"acc={best['best_acc']*100:.1f}%  sparsity={best['sparsity']*100:.1f}%",
        fontsize=12, fontweight="bold")
    axes[0].hist(gates, bins=150, color="#2563eb", edgecolor="none")
    axes[0].axvline(1e-2, color="red", lw=1.5, ls="--", label="Prune threshold 1e-2")
    axes[0].set_title("Full distribution [0,1]"); axes[0].set_xlabel("Gate value")
    axes[0].legend(); axes[0].grid(alpha=0.3)
    gz = gates[gates < 0.05]
    axes[1].hist(gz if len(gz) else [0], bins=60, color="#dc2626", edgecolor="none")
    axes[1].set_title(f"Pruned region  gates < 0.05  ({100*(gates<0.05).mean():.1f}%)")
    axes[1].set_xlabel("Gate value"); axes[1].grid(alpha=0.3)
    ga = gates[gates > 0.5]
    axes[2].hist(ga if len(ga) else [1], bins=60, color="#16a34a", edgecolor="none")
    axes[2].set_title(f"Active region  gates > 0.5  ({100*(gates>0.5).mean():.1f}%)")
    axes[2].set_xlabel("Gate value"); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "gate_distribution.png"), dpi=150); plt.close()

    # ── Trade-off (task spec requirement) ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    lams  = [r["lam"]      for r in results]
    accs  = [r["best_acc"] * 100 for r in results]
    spars = [r["sparsity"] * 100 for r in results]
    ax.plot(lams, accs, "o-", color="#2563eb", lw=2, ms=9, label="Best Test Acc (%)")
    for l, a in zip(lams, accs):
        ax.annotate(f"{a:.1f}%", (l, a), textcoords="offset points",
                    xytext=(0, 9), ha="center", fontsize=10)
    ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("Acc (%)", color="#2563eb")
    ax.tick_params(axis="y", colors="#2563eb")
    ax2 = ax.twinx()
    ax2.plot(lams, spars, "s--", color="#dc2626", lw=2, ms=9, label="Sparsity (%)")
    for l, s in zip(lams, spars):
        ax2.annotate(f"{s:.1f}%", (l, s), textcoords="offset points",
                     xytext=(0,-14), ha="center", fontsize=10, color="#dc2626")
    ax2.set_ylabel("Sparsity (%)", color="#dc2626")
    ax2.tick_params(axis="y", colors="#dc2626")
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc="center left", fontsize=9)
    ax.set_title("λ Trade-off: Accuracy vs Sparsity", fontweight="bold")
    ax.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "tradeoff.png"), dpi=150); plt.close()

    # ── Training curves ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Training Curves — All λ Values", fontsize=13, fontweight="bold")
    keys   = ["test_acc", "sparsity", "gate_mean", "cls_loss", "sp_loss", "temp"]
    titles = ["Test Accuracy","Sparsity @T=15","Gate Mean","CE Loss","Sparsity Loss","Temperature T"]
    ylbls  = ["%", "%", "value", "loss", "mean gate", "T"]
    scales = [100, 100, 1, 1, 1, 1]

    for i, r in enumerate(results):
        c = pal[i]; lbl = f"λ={r['lam']}"
        ep = range(1, len(r["history"]["test_acc"]) + 1)
        for ax, key, sc in zip(axes.ravel(), keys, scales):
            vals = [v * sc for v in r["history"][key]]
            ax.plot(ep, vals, c, lw=1.8, label=lbl)

    for ax, title, yl in zip(axes.ravel(), titles, ylbls):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150); plt.close()

    # ── Per-layer sparsity bar chart ───────────────────────────────────────────
    n_layers = len(results[0]["layer_sp"])
    x, w = np.arange(n_layers), 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, r in enumerate(results):
        vals = [v * 100 for v in r["layer_sp"].values()]
        ax.bar(x + i*w, vals, w, label=f"λ={r['lam']}", color=pal[i], alpha=0.85)
    ax.set_xticks(x + w); ax.set_xticklabels([f"L{j+1}" for j in range(n_layers)])
    ax.set_ylabel("Sparsity (%)"); ax.set_title("Per-Layer Sparsity", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "layer_sparsity.png"), dpi=150); plt.close()

    print(f"  [Plots] Saved to {save_dir}/  (gate_distribution, tradeoff, training_curves, layer_sparsity)")


# ─────────────────────────────────────────────────────────────────────────────
# 10. Summary table
# ─────────────────────────────────────────────────────────────────────────────
def print_table(results):
    print(f"\n{'='*72}")
    print(f"  RESULTS SUMMARY  (task spec table)")
    print(f"{'='*72}")
    print(f"  {'Lambda':>8} | {'Test Acc (%)':>13} | {'Sparsity (%)':>13} | {'Gate Mean':>10}")
    print(f"  {'─'*8}-+-{'─'*13}-+-{'─'*13}-+-{'─'*10}")
    for r in results:
        print(f"  {r['lam']:>8} | {r['best_acc']*100:>12.2f}% | "
              f"{r['sparsity']*100:>12.2f}% | {r['gates'].mean():>10.4f}")
    print(f"{'='*72}")
    print(f"\n  Saved checkpoints:")
    for r in results:
        print(f"    λ={r['lam']}  →  {r['best_path']}  (best)")
        print(f"         →  {r['final_path']}  (final)")


# ─────────────────────────────────────────────────────────────────────────────
# 11. Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    LAMBDAS    = [0.3, 1.0, 3.0]   # low / medium / high  (task spec: 3 values)
    EPOCHS     = 25                 # 25 per run → ~8 min/run on T4
    LR         = 4e-3
    GATE_LR    = 1e-2
    LAM_WARMUP = 4
    BATCH_SIZE = 256
    SAVE_DIR   = "./outputs_model2"

    print(f"  Plan: {len(LAMBDAS)} runs × {EPOCHS} epochs = {len(LAMBDAS)*EPOCHS} total epochs")
    print(f"  ETA on T4: ~{len(LAMBDAS)*EPOCHS//4} min  |  λ values: {LAMBDAS}\n")

    results = []
    for lam in LAMBDAS:
        r = run(lam=lam, epochs=EPOCHS, lr=LR, gate_lr=GATE_LR,
                lam_warmup=LAM_WARMUP, batch_size=BATCH_SIZE,
                save_dir=os.path.join(SAVE_DIR, "ckpts"))
        results.append(r)

    print_table(results)

    # Best = highest accuracy with sparsity > 15%
    candidates = [r for r in results if r["sparsity"] > 0.15]
    best = (max(candidates, key=lambda r: r["best_acc"])
            if candidates else max(results, key=lambda r: r["best_acc"]))

    plot_all(results, best, save_dir=SAVE_DIR)

    # ── Load + verify demo ────────────────────────────────────────────────────
    print(f"\n  ── Verify: loading best checkpoint (λ={best['lam']}) ──")
    m, meta = load_model(best["best_path"])
    m.set_temperature(15.)
    _, test_loader = get_loaders(BATCH_SIZE)
    v_acc = evaluate(m, test_loader)
    print(f"  Loaded model accuracy: {v_acc*100:.2f}%  "
          f"(saved as {meta['test_acc']*100:.2f}%)")
    print(f"\n  Done! All outputs in {SAVE_DIR}/\n")


  Self-Pruning Neural Network  —  CIFAR-10
  Device  : cuda
  GPU     : Tesla T4
  Epochs  : 25 per run  (3 runs total = 75 epochs)
  Method  : Sigmoid gates + L1 sparsity + Temp annealing

  Plan: 3 runs × 25 epochs = 75 total epochs
  ETA on T4: ~18 min  |  λ values: [0.3, 1.0, 3.0]


─────────────────────────────────────────────────────────────────
  λ=0.3  |  25 epochs  |  lr=0.004  gate_lr=0.01
  T: 1→15  |  λ warmup=4ep  |  Mixup + LabelSmooth
─────────────────────────────────────────────────────────────────


100%|██████████| 170M/170M [00:04<00:00, 41.6MB/s]


  Params: 3,549,450  |  Gate params: 902,400  |  Prunable layers: 9

  Ep 1/25 [196/196] loss=2.0982  cls=2.0610  sp=0.4953  acc=20.3%
  ▶ Ep 1/25  train=20.3%  test=35.7%  sparsity=1.0%  gate_mean=0.400  neg_scores=66%  T=1.0  λ=0.075  t=33s
    ★ New best  acc=35.74%  sparsity=1.0%
  Ep 2/25 [196/196] loss=2.0628  cls=1.9912  sp=0.4777  acc=20.1%
  ▶ Ep 2/25  train=20.1%  test=37.4%  sparsity=13.1%  gate_mean=0.258  neg_scores=81%  T=1.1  λ=0.150  t=66s
    ★ New best  acc=37.39%  sparsity=13.1%
  Ep 3/25 [196/196] loss=2.0457  cls=1.9478  sp=0.4349  acc=23.9%
  ▶ Ep 3/25  train=23.9%  test=39.1%  sparsity=41.4%  gate_mean=0.125  neg_scores=92%  T=1.3  λ=0.225  t=99s
    ★ New best  acc=39.06%  sparsity=41.4%
  Ep 4/25 [196/196] loss=2.0506  cls=1.9426  sp=0.3601  acc=23.2%
  ▶ Ep 4/25  train=23.2%  test=41.2%  sparsity=72.6%  gate_mean=0.051  neg_scores=97%  T=1.4  λ=0.300  t=132s
    ★ New best  acc=41.17%  sparsity=72.6%
  Ep 5/25 [196/196] loss=2.0027  cls=1.9215  sp=0.2706  acc=